# 17.15 品質與行為保留了多少？


量化前後平均分數只差0.005，是否就能放心說模型行為相同？如果兩個候選原本差距更小，這點誤差足以換掉第一名，甚至讓回答提早結束。前置是[權重誤差的輸出影響](https://birdhackor.github.io/tiny-perceptron-vlm/17.3.html)與[下一字分數](https://birdhackor.github.io/tiny-perceptron-vlm/4.6.html)：模型的logits是各候選的未正規化分數，greedy生成直接取最大分數者，並不根據平均誤差決定答案。

假設候選只有EOS與「貓」。EOS是[結束符號](https://birdhackor.github.io/tiny-perceptron-vlm/7.9.html)，選中它代表回答停止。原分數 `[1,1.005]`讓「貓」勝出；量化後只有第一項增加0.01，變 `[1.01,1.005]`，現在EOS勝出。平均絕對差是 `(0.01+0)/2=0.005`，很小，但可觀察的行為從繼續寫變成結束。下一步的輸入也會跟著改變，所以差異可能沿生成過程擴大。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
import torch

tokens = ["<EOS>", "貓"]
original = torch.tensor([1.0, 1.005])
quantized = torch.tensor([1.01, 1.005])
mae = (original - quantized).abs().mean().item()
print("分數MAE", round(mae, 4))
print("原版選擇", tokens[original.argmax().item()])
print("量化版選擇", tokens[quantized.argmax().item()])

輸入兩個固定分數向量，只模擬一個位置的邊界，不是整個語言模型。`argmax`回傳最大值的索引，`.item()`取得整數，再查tokens顯示候選文字。輸出應為0.005、「貓」與「<EOS>」。這證明小數值差不能保證離散選擇不變，沒有證明實際量化模型一定會提早結束；真正行為必須透過生成來觀察。

評估時讓原版與量化版使用同一批提示、相同輸入處理與生成設定。先核對[跨切詞可比較的BPB](https://birdhackor.github.io/tiny-perceptron-vlm/6.5.html)：它把負對數機率換算成每個原始byte的平均位元成本，讓不同token長度的預測代價可比較。再看任務正確率與具體輸出，不只看訓練誤差或分數MAE。

格式检查可問JSON是否可解析、回答是否完整保留必要欄位、結束符號是否使用合適；誠實检查可問缺乏資訊時是否說明不確定，而不是更有自信地編答案；安全與拒絕检查則重跑既有情境，看哪些原來合適的回應改變。應按類別列差異，而不是用一個平均高分掩蓋少數明顯退步。這些結论只涵蓋實際測試情境，不需在此假設全面保證。

同時量檔案大小與最高記憶體，並分開記錄[prefill與decode時間](https://birdhackor.github.io/tiny-perceptron-vlm/16.2.html)：prefill是先讀取完整提示的計算，decode是之後逐步產生新文字的計算，兩段各自計時，才能看出成本改在哪裡。若用抽樣生成，隨機種子是決定隨機數起點的設定；例如用 `torch.manual_seed(0)`固定這個起點，在原版與量化版各次生成之前都重設成0，配對比較同一批提示。還可換成1、2等種子，記錄每組設定與結果差異，而不是只報最好一次。greedy則直接選最高分候選，不抽樣；也要註明這項設定，避免把抽樣的偶然變化全算到量化。

練習只把原版第二項1.005改成2，量化版也同步改成2。先預測MAE仍0.005，但兩版都選「貓」，再執行核對。相同誤差在不同排名餘裕下可能有不同後果，這正是平均數字與實際行為都要檢查的原因。
